In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
import joblib

# 1. Загрузка и первичная очистка
df = pd.read_csv('foodAllergyAnalysisZenodo.csv')

# Создаем бинарные признаки для сопутствующих заболеваний (Атопический марш)
df['HAS_ASTHMA'] = df['ASTHMA_START'].notna().astype(int)
df['HAS_ECZEMA'] = df['ATOPIC_DERM_START'].notna().astype(int)

# 2. Определение признаков (Features) и целевых переменных (Targets)
feature_cols = [
    'BIRTH_YEAR', 'GENDER_FACTOR', 'RACE_FACTOR', 
    'ETHNICITY_FACTOR', 'HAS_ASTHMA', 'HAS_ECZEMA'
]

# Список всех аллергий, которые мы хотим прогнозировать
target_cols = [
    'PEANUT_ALG_START', 'MILK_ALG_START', 'EGG_ALG_START',
    'SHELLFISH_ALG_START', 'SOY_ALG_START', 'WHEAT_ALG_START'
]

# Подготовка X (превращаем категории в числа)
X = pd.get_dummies(df[feature_cols], drop_first=True)
X_columns = X.columns # Сохраняем порядок колонок для приложения

models = {}

print("Обучение моделей...")

for col in target_cols:
    allergy_name = col.replace('_ALG_START', '').lower()
    
    # Целевая переменная: 1 если есть дата начала аллергии, иначе 0
    y = df[col].notna().astype(int)
    
    # Разделение с учетом дисбаланса классов (stratify)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Модель с балансировкой весов (так как аллергиков мало)
    model = RandomForestClassifier(
        n_estimators=100, 
        class_weight='balanced', 
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    # Оценка
    probs = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    print(f"Аллергия: {allergy_name:10} | AUC-ROC: {auc:.4f}")
    
    models[allergy_name] = model

# 3. Сохранение для приложения
# Сохраняем словарь моделей и список колонок, чтобы приложение знало порядок признаков
joblib.dump({'models': models, 'X_columns': X_columns}, 'allergy_predictor_v1.pkl')
print("\nМодели сохранены в файл allergy_predictor_v1.pkl")

# 4. Функция для интерфейса будущего приложения
def predict_risks(user_input):
    """
    user_input: dict с данными пациента
    """
    input_df = pd.DataFrame([user_input])
    input_encoded = pd.get_dummies(input_df).reindex(columns=X_columns, fill_value=0)
    
    predictions = {}
    for name, model in models.items():
        prob = model.predict_proba(input_encoded)[0][1]
        predictions[f"{name}_risk"] = round(float(prob) * 100, 2) # в процентах
        
    return predictions

# Пример теста
example_user = {
    'BIRTH_YEAR': 2015,
    'GENDER_FACTOR': 'S1 - Female',
    'RACE_FACTOR': 'R0 - White',
    'ETHNICITY_FACTOR': 'E0 - Non-Hispanic',
    'HAS_ASTHMA': 1,
    'HAS_ECZEMA': 0
}

print("\nРезультат для примера:")
print(predict_risks(example_user))

Обучение моделей...
Аллергия: peanut     | AUC-ROC: 0.7243
Аллергия: milk       | AUC-ROC: 0.6938
Аллергия: egg        | AUC-ROC: 0.7624
Аллергия: shellfish  | AUC-ROC: 0.7071
Аллергия: soy        | AUC-ROC: 0.6941
Аллергия: wheat      | AUC-ROC: 0.6624

Модели сохранены в файл allergy_predictor_v1.pkl

Результат для примера:
{'peanut_risk': 27.87, 'milk_risk': 61.39, 'egg_risk': 0.0, 'shellfish_risk': 0.0, 'soy_risk': 44.99, 'wheat_risk': 0.0}
